In [3]:
from ib_insync import *
import nest_asyncio
import pandas as pd
import numpy as numpy
import time

In [8]:
nest_asyncio.apply()

ib = IB()
ib.connect('127.0.0.1', 7496, clientId=4)

Error 321, reqId -1: Error validating request.-'cp' : cause - The TWS/Gateway process is running in Read-Only mode. Trading actions are not allowed.
Error 321, reqId -1: Error validating request.-'bZ' : cause - The TWS/Gateway process is running in Read-Only mode. Trading actions are not allowed.
open orders request timed out
completed orders request timed out


<IB connected to 127.0.0.1:7496 clientId=4>

In [10]:
contract = Stock('XLE', 'SMART', 'USD')

bars = ib.reqHistoricalData(
    contract,
    endDateTime='',
    durationStr='20 D',
    barSizeSetting='30 mins',
    whatToShow='TRADES',
    useRTH=True
)

df = util.df(bars)
df['symbol'] = contract.symbol
df.set_index('date', inplace=True)

# print(df.tail())

In [15]:
# File Paths
ticker_sector_map_file_path = r".\inputs\ticker_sector_map.csv"

In [16]:
# Read Input Files
ticker_sector_map_df = pd.read_csv(ticker_sector_map_file_path)

## Data

### Getting all tickers 

In [85]:
market_cap_threshold = 5000000000  # 1 Billion
ticker_sector_map_condition = (~(ticker_sector_map_df["Sector"].isna())) & (ticker_sector_map_df["Market Cap"] != 0) & (ticker_sector_map_df["Last Sale"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float) > 1) & (ticker_sector_map_df["Market Cap"] > market_cap_threshold)

tickers = ticker_sector_map_df[ticker_sector_map_condition]["Symbol"].tolist()
ipo_tickers = ticker_sector_map_df[ticker_sector_map_df["IPO Year"].fillna(0).astype(int) == 2026]["Symbol"].tolist()
all_tickers = list(set(tickers + ipo_tickers))